In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE = "hidden"
os.makedirs(DRIVE, exist_ok=True)
print(os.listdir(DRIVE))

Mounted at /content/drive
[]


In [2]:
BASE = "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads"

!wget -q --show-progress -O /content/book_id_map.csv {BASE}/book_id_map.csv
!ls -lh /content/book_id_map.csv

/content/book_id_ma 100%[===================>]  36.09M  84.2MB/s    in 0.4s    
-rw-r--r-- 1 root root 37M Jan 17  2025 /content/book_id_map.csv


In [3]:
import pandas as pd

id_map = pd.read_csv("/content/book_id_map.csv")
print(id_map.shape, id_map.dtypes.to_dict())
id_map.head()

(2360650, 2) {'book_id_csv': dtype('int64'), 'book_id': dtype('int64')}


,book_id_csv,book_id
0,0,34684622
1,1,34536488
2,2,34017076
3,3,71730
4,4,30422361


In [4]:
!wget -q --show-progress -O /content/goodreads_interactions.csv {BASE}/goodreads_interactions.csv
!head -3 /content/goodreads_interactions.csv

/content/goodreads_ 100%[===================>]   4.02G   103MB/s    in 56s     
user_id,book_id,is_read,rating,is_reviewed
0,948,1,5,0
0,947,1,5,1


In [7]:
catalog = pd.read_parquet("/content/catalog_ids.parquet")

e2w = (catalog[["work_id", "book_id_all"]]
       .explode("book_id_all")
       .dropna(subset=["book_id_all"]))
e2w["book_id"] = e2w.book_id_all.astype(str)
e2w = e2w[["book_id", "work_id"]]

print(f"{len(catalog):,} works, {len(e2w):,} editions")

1,002,607 works, 1,760,370 editions


In [8]:
id_map["book_id"] = id_map.book_id.astype(str)
c2w = id_map.merge(e2w, on="book_id", how="inner")[["book_id_csv", "work_id"]]

KEEP = dict(zip(c2w.book_id_csv, c2w.work_id))
print(f"{len(KEEP):,} of {len(id_map):,} ids are in the catalog")

del e2w, c2w, catalog

1,760,366 of 2,360,650 ids are in the catalog


In [9]:
import pyarrow as pa, pyarrow.parquet as pq
from tqdm.auto import tqdm

OUT = f"{DRIVE}/interactions.parquet"
DROP_UNREAD = True

writer, seen, kept = None, 0, 0

for part in tqdm(pd.read_csv("/content/goodreads_interactions.csv", chunksize=5_000_000)):
    seen += len(part)
    if DROP_UNREAD and "is_read" in part.columns:
        part = part[part.is_read == 1]
    part = part[part.book_id.isin(KEEP)]
    if part.empty:
        continue
    part["work_id"] = part.book_id.map(KEEP)
    part = part.drop(columns=["book_id"])

    t = pa.Table.from_pandas(part, preserve_index=False)
    writer = writer or pq.ParquetWriter(OUT, t.schema, compression="zstd")
    writer.write_table(t)
    kept += len(part)

writer.close()
print(f"seen {seen:,}  kept {kept:,}  ({kept/seen:.1%})")

0it [00:00, ?it/s]

seen 228,648,342  kept 105,744,280  (46.2%)


In [10]:
inter = pd.read_parquet(OUT)

print(f"{len(inter):,} interactions")
print(f"{inter.user_id.nunique():,} users")
print(f"{inter.work_id.nunique():,} works")

per_user = inter.groupby("user_id").size()
print(f"\nusers with >=10: {per_user.ge(10).sum():,} ({per_user.ge(10).mean():.1%})")
print(per_user.describe().round(1).to_string())

105,744,280 interactions
827,743 users
996,177 works

users with >=10: 700,843 (84.7%)
count    827743.0
mean        127.8
std         235.2
min           1.0
25%          21.0
50%          54.0
75%         141.0
max       36069.0


In [11]:
!ls -lh /content/*.parquet /content/drive/MyDrive/book-rec/*.parquet 2>/dev/null

-rw-r--r-- 1 root root  20M Sep  1 21:20 /content/catalog_ids.parquet
-rw------- 1 root root 424M Sep  1 21:25 /content/drive/MyDrive/book-rec/interactions.parquet


In [12]:
import os
print(f"{os.path.getsize(OUT) / 1e6:,.0f} MB")

444 MB


In [10]:
from google.colab import files
files.download("/content/interactions.parquet")